<a href="https://colab.research.google.com/github/mritunjay29-ai/Namekart-Domain-Name-Evaluation/blob/main/category_find_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from transformers import TFDistilBertModel, DistilBertTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

file_path = "/content/Model training data.csv"
df = pd.read_csv(file_path)
label_encoder = LabelEncoder()
df['category_encoded'] = label_encoder.fit_transform(df['category'])

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['domain'].astype(str).tolist(), df['category_encoded'].tolist(), test_size=0.1, random_state=42
)

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
max_len = 32
def encode_texts(texts):
    encodings = tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='np'
    )
    return encodings['input_ids'], encodings['attention_mask']

train_input_ids, train_attention_mask = encode_texts(train_texts)
val_input_ids, val_attention_mask = encode_texts(val_texts)
train_labels = np.array(train_labels, dtype=np.int32)
val_labels = np.array(val_labels, dtype=np.int32)

class DistilBertEmbedding(tf.keras.layers.Layer):
    def __init__(self, bert_model_name="distilbert-base-uncased", **kwargs):
        super(DistilBertEmbedding, self).__init__(**kwargs)
        self.bert = TFDistilBertModel.from_pretrained(bert_model_name)

    def call(self, inputs):
        input_ids, attention_mask = inputs
        bert_output = self.bert(input_ids, attention_mask=attention_mask)
        return bert_output.last_hidden_state[:, 0, :]


num_labels = len(label_encoder.classes_)

input_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name='input_ids')
attention_mask = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name='attention_mask')

bert_embedding = DistilBertEmbedding()([input_ids, attention_mask])  # Use Custom Layer
dropout = tf.keras.layers.Dropout(0.3)(bert_embedding)
out = tf.keras.layers.Dense(num_labels, activation='softmax')(dropout)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=out)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

history = model.fit(
    x={'input_ids': train_input_ids, 'attention_mask': train_attention_mask},
    y=train_labels,
    validation_data=({'input_ids': val_input_ids, 'attention_mask': val_attention_mask}, val_labels),
    batch_size=16,
    epochs=5
)

val_loss, val_accuracy = model.evaluate(
    x={'input_ids': val_input_ids, 'attention_mask': val_attention_mask},
    y=val_labels
)

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Loss: {val_loss:.4f}")

model.save_weights("domain_category_classifier_weights.h5")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertModel: ['vocab_layer_norm.weight', 'vocab_layer_norm.bias', 'vocab_projector.bias', 'vocab_transform.bias', 'vocab_transform.weight']
- This IS expected if you are initializing TFDistilBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFDistilBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertModel for predictions without further training.


Epoch 1/5
5078/5078 [==============================] - 370s 69ms/step - loss: 0.5178 - accuracy: 0.7571 - val_loss: 0.4570 - val_accuracy: 0.7997
Epoch 2/5
5078/5078 [==============================] - 358s 71ms/step - loss: 0.3782 - accuracy: 0.8468 - val_loss: 0.4278 - val_accuracy: 0.8240
Epoch 3/5
5078/5078 [==============================] - 361s 71ms/step - loss: 0.2786 - accuracy: 0.8975 - val_loss: 0.4647 - val_accuracy: 0.8230
Epoch 4/5
5078/5078 [==============================] - 352s 69ms/step - loss: 0.2032 - accuracy: 0.9323 - val_loss: 0.5160 - val_accuracy: 0.8262
Epoch 5/5
283/283 [==============================] - 11s 40ms/step - loss: 0.6060 - accuracy: 0.8121
Validation Accuracy: 0.8121
Validation Loss: 0.6060


In [11]:
model.save("domain_category_classifier_tf.h5")


In [12]:
import tensorflow as tf
from transformers import DistilBertTokenizer
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os

model_path = "/content/domain_category_classifier_tf.h5"
if os.path.exists(model_path):
    model = tf.keras.models.load_model(model_path, custom_objects={"DistilBertEmbedding": DistilBertEmbedding})
else:
    raise FileNotFoundError(f"Model file not found at {model_path}")

test_file_path = "/content/Model testing dataset.xlsx"
test_df = pd.read_excel(test_file_path)

if 'domain' not in test_df.columns:
    raise ValueError("Test data must contain a 'domain' column")

train_file_path = "/content/Model training data.csv"  # Update the path
train_df = pd.read_csv(train_file_path)
label_encoder = LabelEncoder()
label_encoder.fit(train_df['category'])  # Ensure consistent encoding

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
max_len = 32

def encode_texts(texts):
    encodings = tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='np'
    )
    return encodings['input_ids'], encodings['attention_mask']

test_input_ids, test_attention_mask = encode_texts(test_df['domain'].astype(str))

predictions = model.predict({'input_ids': test_input_ids, 'attention_mask': test_attention_mask})
predicted_labels = np.argmax(predictions, axis=1)

test_df['predicted_category'] = label_encoder.inverse_transform(predicted_labels)

output_file_path = "test_predictions.csv"
test_df.to_csv(output_file_path, index=False)

print(f"Predictions saved to {output_file_path}")
print(test_df[['domain', 'predicted_category']].head())

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertModel: ['vocab_layer_norm.weight', 'vocab_layer_norm.bias', 'vocab_projector.bias', 'vocab_transform.bias', 'vocab_transform.weight']
- This IS expected if you are initializing TFDistilBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFDistilBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertModel for predictions without further training.


21413/21413 ━━━━━━━━━━━━━━━━━━━━ 597s 28ms/step
Predictions saved to test_predictions.csv
          domain predicted_category
0  altavista.com           specific
1      bingo.com           specific
2        fly.com           specific
3      autos.com           specific
4    england.com           specific
